In [1]:
# ===========================================
# 청팀을 더 자주 응원하는 AI
# ===========================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from IPython.display import display, HTML
import ipywidgets as wg

# ----- 구글 시트 데이터 로드 / 요약 리포트 -----
df = pd.read_csv("https://docs.google.com/spreadsheets/d/1SyBg64IX_uX290hsIwGH3zZwe_Aeb1SBLx146zSGIro/export?format=csv")

total = len(df)
blue = int((df["winner"] == 1).sum())  # 청팀 승 수
red  = int((df["winner"] == 0).sum())  # 홍팀 승 수

print("\n[학습 데이터 요약]")
print(f"- 전체 경기 수: {total:,}")
print(f"- 청팀 승(1): {blue:,}건 ({blue/total*100:.1f}%)")
print(f"- 홍팀 승(0): {red:,}건 ({red/total*100:.1f}%)\n\n")

# ----- 모델 학습 -----
X = df[["fit_b", "fit_r"]].values
y = df["winner"].astype(int).values
model = LogisticRegression(max_iter=2000).fit(X, y)

# ---- 샘플 분포 설정('비슷한 경기'가 많이 생성되도록 표준편차를 살짝 축소) ----
mu_b, sd_b = df["fit_b"].mean(), df["fit_b"].std(ddof=0)
mu_r, sd_r = df["fit_r"].mean(), df["fit_r"].std(ddof=0)
sd_b_s, sd_r_s = max(1e-6, sd_b*0.75), max(1e-6, sd_r*0.75)

# ----- 예측 -----
rng = np.random.default_rng()  # 난수 생성(Random Number Generator)
def _sample_skills():
    return rng.normal(mu_b, sd_b_s), rng.normal(mu_r, sd_r_s)

def predict_once():
    fb, fr = _sample_skills()
    p_b = model.predict_proba([[fb, fr]])[0, 1]
    p_r = 1.0 - p_b

    if np.isclose(p_b, p_r):
        pred_team = rng.choice(['청팀','홍팀'])
    else:
        pred_team = '청팀' if p_b > p_r else '홍팀'

    mis = (fr > fb) and (pred_team == '청팀')
    return dict(pred_team=pred_team, flag='🟦' if pred_team=='청팀' else '🟥',
                prob_b=p_b, prob_r=p_r, fit_b=fb, fit_r=fr, mis=mis)

# ----- 색깔 네모 그리기 함수 정의 -----
DOT, MARGIN = 16, 3  # 네모 한 변의 길이와 점 사이 여백 설정
STYLE = f"""
<style>
.dotboard {{
  font-size:0; line-height:0; padding:8px;  /* 인라인 블록 간 공백 제거, 여백 */
  border:1px solid #ddd; border-radius:10px; max-width:800px;  /* 테두리, 둥근 모서리, 한 줄 최대 폭 */
}}
.dot {{ display:inline-block; width:{DOT}px; height:{DOT}px; border-radius:0; margin:{MARGIN}px; box-sizing:border-box; }}  /* 네모 하나의 크기/여백/박스모델 */
.blue {{ background:#1f77b4; }} .red {{ background:#d62728; }}  /* 청홍 색상 */
</style>
"""
def dot_blue():     return f'<span class="dot blue"></span>'  # 파란 네모 HTML 스니펫 반환
def dot_blue_mis(): return (f'<span style="display:inline-block;width:{DOT}px;height:{DOT}px;'  # 편향적 오판일 때: 파란 바탕 + 빨간 테두리
                             f'border-radius:0;background:#1f77b4;border:2px solid #d62728;box-sizing:border-box;'
                             f'margin:{MARGIN}px;"></span>')
def dot_red():      return f'<span class="dot red"></span>'  # 빨간 네모 HTML 스니펫 반환
display(HTML(STYLE))  # 위에서 정의한 CSS를 노트북 출력 영역에 주입하여 이후 HTML에 스타일 적용

# ----- 예측 결과 누적 집계/시각화 -----
BATCH_N = 100  # 일괄 실행 시 예측 반복 횟수
board = wg.HTML()  # 색깔 네모 보드를 렌더링할 HTML 위젯
out = wg.Output(layout={'border':'1px solid #ddd','padding':'8px'})  # 결과/요약 텍스트가 찍힐 출력 영역

dots_html, flag_seq = [], []  # 색깔 네모 HTML 스니펫 누적 리스트, 이모지(🟦/🟥) 시퀀스
count_blue = count_red = mis_count = 0  # 청/홍/편향오판 누적 카운터

def _reset_state():
    global dots_html, flag_seq, count_blue, count_red, mis_count  # 함수 내에서 전역 상태 갱신
    dots_html, flag_seq = [], []  # 색깔 네모/이모지 이력 초기화
    count_blue = count_red = mis_count = 0  # 카운터 초기화
    board.value = '<div class="dotboard"></div>'  # 빈 보드로 초기 렌더(틀만 표시)

def render_board():  # 범례: 청/홍/편향오판 표시 설명
    legend = (f'<div style="font-size:12px;color:#555;margin:4px 8px">'
              f'{dot_blue()} 청팀 &nbsp; {dot_red()} 홍팀 &nbsp; {dot_blue_mis()} 편향적 오판</div>')
    board.value = legend + '<div class="dotboard">' + ''.join(dots_html) + '</div>'  # 범례 + 누적 네모들을 하나로 합쳐 보드에 세팅

def render_summary():
    total = len(flag_seq) or 1  # 0으로 나누기 방지(아직 예측 0건일 때 1로 처리)
    b_rate = (count_blue/total)*100  # 청팀 예측 비율(%)
    r_rate = (count_red /total)*100  # 홍팀 예측 비율(%)
    mis_rate = (mis_count/total)*100  # 편향적 오판 비율(%)
    return HTML(f"""
    <div style="font-family:system-ui,Apple SD Gothic Neo,sans-serif; line-height:1.6">
      <h4 style="margin:4px 0"><br>누적 예측 결과 (🟦=청팀, 🟥=홍팀)</h4>
      <p style="margin:6px 0 0 0">누적 경기 수: <b>{len(flag_seq)}</b></p>  <!-- 총 예측 건수 -->
      <ul style="margin:4px 0 0 16px">
        <li>청팀(🟦) 비율: <b>{b_rate:.1f}%</b> (건수 {count_blue})</li>  <!-- 청팀 예측 횟수/비율 -->
        <li>홍팀(🟥) 비율: <b>{r_rate:.1f}%</b> (건수 {count_red})</li>    <!-- 홍팀 예측 횟수/비율 -->
        <li>편향적 오판({dot_blue_mis()}): <b>{mis_rate:.1f}%</b> (건수 {mis_count})</li>  <!-- 오판(청 예측 & 홍 우세) 횟수/비율 -->
      </ul>
    </div>
    """)

def apply_result(res, acc):
    global count_blue, count_red, mis_count  # 전역 카운터 갱신
    if res['pred_team'] == '청팀':  # 이번 예측이 청팀이라면
        dots_html.append(dot_blue_mis() if res['mis'] else dot_blue())  # 오판이면 빨간테두리-파란 네모, 아니면 파란 네모
        count_blue += 1; acc['b'] += 1  # 누적/배치 카운터 동시 증가
    else:
        dots_html.append(dot_red()); count_red += 1; acc['r'] += 1  # 홍팀 네모 추가 및 카운터 증가
    mis_count += int(res['mis'])  # True→1, False→0 으로 변환하여 오판 누적
    flag_seq.append(res['flag'])  # 🟦/🟥 이모지 이력 추가(요약 표시용)

# ----- 버튼 클릭 이벤트 함수 정의 -----  # 단일 실행/일괄 실행/초기화 버튼에 연결될 콜백들
@out.capture(clear_output=True)  # 이 함수 내 출력/예외를 out 위젯에 캡처, 호출마다 이전 출력 지움
def on_one(_):
    acc = {'b':0,'r':0}  # 이번 클릭(단일) 동안의 청/홍 예측 집계(배치 요약 메시지용)
    res = predict_once()  # 한 경기 샘플링→확률 계산→예측→오판 판정 결과 딕셔너리
    apply_result(res, acc)  # 상태(네모/카운터/이모지) 반영 + 배치 집계 갱신
    display(HTML(
        f"<h4 style='margin:0 0 6px 0'>이번 경기 예측: {res['flag']} ({res['pred_team']} 승)</h4>"  # 한 줄 요약(깃발+팀)
        f"<p style='margin:0;font-size:13px;color:#555;'>"
        f"승리 예측 확률값: 청팀={res['prob_b']:.3f}, 홍팀={res['prob_r']:.3f} <br> "  # 모델 예측 확률(청/홍)
        f"실력 점수: 청팀={res['fit_b']:.1f}, 홍팀={res['fit_r']:.1f}</p>"  # 이번 샘플에서의 실력 입력값
    ))
    display(render_summary()); render_board()  # 누적 요약을 out에 표시, 보드 HTML을 갱신

@out.capture(clear_output=True)  # 일괄 실행도 동일하게 out에 출력 캡처 및 덮어쓰기
def on_batch(_):
    acc = {'b':0,'r':0}  # 이번 배치에서의 청/홍 건수 집계
    for _ in range(BATCH_N):  # 설정된 횟수만큼 반복 예측
        apply_result(predict_once(), acc)  # 예측→상태 반영을 루프 내에서 누적
    display(HTML(f"<h4 style='margin:0 0 6px 0'>일괄 {BATCH_N}회 완료</h4>"  # 배치 완료 요약(건수만 간단 표기)
                 f"<p style='margin:0;font-size:13px;color:#555;'>승리 예측: 청팀 {acc['b']}회, 홍팀 {acc['r']}회</p>"))
    display(render_summary()); render_board()  # 전체 누적 요약과 보드 재렌더

@out.capture(clear_output=True)  # 초기화 시에도 출력 정리
def on_clear(_):
    _reset_state()  # 누적 상태 초기화(점/카운터/플래그 비우기)
    display(HTML("<b>🔄 초기화 완료</b>"))  # 초기화 알림
    display(render_summary())  # 빈 상태 기준 요약(0건) 표시

# ----- UI/시작 -----  # 버튼 생성/이벤트 바인딩 및 초기 화면 구성
btn_one   = wg.Button(description="단일 테스트", layout=wg.Layout(width='140px'))  # 단일 실행 버튼
btn_many  = wg.Button(description=f"일괄 {BATCH_N}회", layout=wg.Layout(width='140px'))  # 배치 실행 버튼
btn_clear = wg.Button(description="초기화", button_style='danger', layout=wg.Layout(width='100px'))  # 상태 초기화 버튼
btn_one.on_click(on_one); btn_many.on_click(on_batch); btn_clear.on_click(on_clear)  # 버튼과 콜백 연결

_reset_state()  # 시작 시 깨끗한 상태로 보드/카운터 초기화
display(wg.HBox([btn_one, btn_many, btn_clear]))  # 버튼 3개를 가로로 배치하여 렌더
display(out); display(board)  # 출력 영역(out)과 점보드(board) 위젯을 화면에 표시



[학습 데이터 요약]
- 전체 경기 수: 1,200
- 청팀 승(1): 810건 (67.5%)
- 홍팀 승(0): 390건 (32.5%)




Output(layout=Layout(border='1px solid #ddd', padding='8px'))

HTML(value='<div class="dotboard"></div>')